In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load Dataset
df = pd.read_csv("uber.csv")

print("First 5 rows:")
print(df.head())

# 2. Data Preprocessing

# Remove unnecessary column
df.drop(columns=["Unnamed: 0", "key"], inplace=True, errors="ignore")

# Remove missing values
df.dropna(inplace=True)

# Convert pickup date to datetime
df["pickup_datetime"] = pd.to_datetime(
    df["pickup_datetime"], errors="coerce", utc=True
)

df.dropna(inplace=True)

# Extract useful date features
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month

# Drop original datetime column
df.drop(columns=["pickup_datetime"], inplace=True)

print("\nAfter preprocessing:")
print(df.head())

# 3. Identify and Remove Outliers using IQR

columns = ["fare_amount", "passenger_count"]

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

    

# Remove invalid geographic coordinates
df = df[
    df["pickup_longitude"].between(-180, 180) &
    df["dropoff_longitude"].between(-180, 180) &
    df["pickup_latitude"].between(-90, 90) &
    df["dropoff_latitude"].between(-90, 90)
]

# Remove non-positive fares
df = df[df["fare_amount"] > 0]

print("\nDataset after removing outliers:")
print(df.shape)

# 4. Correlation Analysis

correlation = df.corr(numeric_only=True)

print("\nCorrelation Matrix:")
print(correlation)

plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# 5. Separate Features and Target

X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]

# 6. Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_r2 = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

# 8. Random Forest Regression

rf = RandomForestRegressor()

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

# 9. Compare Results

print("\n----- MODEL COMPARISON -----")

print("\nLinear Regression")
print("R2 Score:", lr_r2)
print("RMSE:", lr_rmse)

print("\nRandom Forest Regression")
print("R2 Score:", rf_r2)
print("RMSE:", rf_rmse)

# 10. Display Comparison

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "R2 Score": [lr_r2, rf_r2],
    "RMSE": [lr_rmse, rf_rmse]
})

print("\nFinal Comparison:")
print(results)

# 11. Visualize Model Performance

results.set_index("Model")[["R2 Score", "RMSE"]].plot(
    kind="bar", figsize=(8, 5)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()
